# 05 Model Evaluation

Audit saved predictions and summarize historical-window performance. This notebook does not train models. Run notebooks 02-04 first to produce experiment records.


## 1. Setup
Set the data path, choose a runtime and install the dependencies used below. The `quick` profile checks the workflow with a small budget; use `report` for the report experiments. Place the four original CSV files in one folder.


In [ ]:
from pathlib import Path
# Use the same data directory and run name in all notebooks.
PROFILE = globals().get('PROFILE', 'quick')  # quick: smoke test; report: full training
DATA_SOURCE = globals().get('DATA_SOURCE', 'drive')  # drive, upload or local
DATA_DIR = Path(globals().get('DATA_DIR', '/content/drive/MyDrive/bt-4012-competition-2026'))
OUTPUT_ROOT = Path(globals().get('OUTPUT_ROOT', '/content/drive/MyDrive/BT4012_Final_Report_English'))
RUN_NAME = globals().get('RUN_NAME', 'final_report_english_v1')
INSTALL_DEPENDENCIES = globals().get('INSTALL_DEPENDENCIES', True)
EXPECTED_COMPETITION = globals().get('EXPECTED_COMPETITION', True)
assert PROFILE in ['quick', 'report']
OUT = OUTPUT_ROOT / RUN_NAME / PROFILE


In [ ]:
import sys, subprocess
# Keep the torch/CUDA version supplied by Colab.
PACKAGES = ['numpy>=1.26', 'pandas>=2.0', 'scipy>=1.11', 'scikit-learn>=1.6', 'joblib']
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES], check=True)


In [ ]:
CSV_FILES = ['train.csv', 'test.csv', 'txs_edgelist.csv', 'sample_submission.csv']

if DATA_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
elif DATA_SOURCE == 'upload':
    from google.colab import files
    DATA_DIR = Path('/content/bt4012_data')
    DATA_DIR.mkdir(exist_ok=True)
    missing = [name for name in CSV_FILES if not (DATA_DIR / name).exists()]
    if missing:
        print('Upload the missing files:', missing)
        for name, content in files.upload().items():
            if name not in missing:
                raise ValueError('Unexpected file: ' + name)
            (DATA_DIR / name).write_bytes(content)
    if str(OUTPUT_ROOT).startswith('/content/drive/'):
        OUTPUT_ROOT = Path('/content/BT4012_Final_Report_English')
        OUT = OUTPUT_ROOT / RUN_NAME / PROFILE
elif DATA_SOURCE != 'local':
    raise ValueError(DATA_SOURCE)
for name in CSV_FILES:
    if not (DATA_DIR / name).is_file():
        raise FileNotFoundError(DATA_DIR / name)
OUT.mkdir(parents=True, exist_ok=True)
print('Mode:', PROFILE, '| Output:', OUT)


## 2. Required functions
Only the data structures, models and helpers used by this notebook are included below.


In [ ]:
from pathlib import Path
from sklearn.metrics import average_precision_score
import hashlib
import json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

VERSION = 'bt4012-colab-report-20261021-v1'

ENGINE_HASH = '3661498eb4e7e545253aeae84b480c6d4c349dc7e125920bf97989a5e96c0840'

def serial(x):
    if isinstance(x, (np.integer, np.floating)):
        return x.item()
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, Path):
        return str(x)
    raise TypeError(type(x).__name__)

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=serial, allow_nan=False))
    temp.replace(path)

def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=serial).encode()).hexdigest()

def array_hash(x):
    return hashlib.sha256(np.ascontiguousarray(x).tobytes()).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 ** 2), b''):
            h.update(block)
    return h.hexdigest()

def metrics(y, p):
    y = np.asarray(y)
    p = np.asarray(p)
    if len(y) != len(p) or not len(y) or (not np.isfinite(p).all()):
        raise ValueError('Empty, nonfinite or unaligned predictions')
    if not np.isin(y, [0, 1]).all():
        raise ValueError('Unknown labels cannot be scored')
    return dict(n=len(y), positives=int(y.sum()), auc=float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else None, ap=float(average_precision_score(y, p)) if len(np.unique(y)) == 2 else None)

def profiles(mode):
    """Use a small budget for a smoke test or the report budget for full experiments."""
    if mode not in ['quick', 'report']:
        raise ValueError(mode)
    return dict(mode=mode, seeds=[42] if mode == 'quick' else [17, 42, 73], cutoffs=[21] if mode == 'quick' else [7, 10, 14, 17, 21], max_epochs=2 if mode == 'quick' else 100, patience=1 if mode == 'quick' else 10, trees=30 if mode == 'quick' else None, batch_size=256)

def prediction_records(out, stage=None):
    records = []
    for f in Path(out).rglob('complete.json'):
        r = json.loads(f.read_text())
        if all((k in r for k in ['spec', 'path', 'metrics', 'query_ids_hash'])) and 'stage' in r['spec'] and (r['spec'].get('engine_hash') == globals().get('ENGINE_HASH', VERSION)) and (stage is None or r['spec']['stage'] == stage):
            records.append(r)
    return records

def aggregate_runs(records):
    """Average seed predictions before calculating each window metric."""
    groups = {}
    for r in records:
        s = r['spec']
        g = {k: v for k, v in s.items() if k != 'seed'}
        key = fingerprint(g)
        groups.setdefault(key, (g, []))[1].append(r)
    rows = []
    for spec, rs in groups.values():
        rs = sorted(rs, key=lambda r: r['spec']['seed'])
        seeds = [r['spec']['seed'] for r in rs]
        if len(seeds) != len(set(seeds)):
            raise ValueError('Duplicate seeded trials')
        expected = spec['profile']['seeds']
        complete = sorted(seeds) == sorted(expected)
        reference = None
        parts = []
        for r in rs:
            frame = pd.read_csv(Path(r['path']) / 'predictions.csv', float_precision='round_trip')
            if reference is None:
                reference = frame
            else:
                np.testing.assert_array_equal(reference[['txId', 'time_step', 'label']], frame[['txId', 'time_step', 'label']])
            parts.append(frame.prediction.to_numpy())
        bag = np.column_stack(parts).mean(1)
        row = dict(stage=spec['stage'], model=spec['model'], variant=spec['variant'], cut=spec['cut'], horizon=spec['horizon'], kind=spec['kind'], config_id=fingerprint(spec['config'])[:10], inner_horizon=spec['inner_horizon'], scaler=spec['scaler'], base=spec['base'], complete=complete, seeds=seeds, **metrics(reference.label, bag))
        rows.append(row)
    return pd.DataFrame(rows)

def audit_predictions(records):
    rows = []
    for r in records:
        path = Path(r['path']) / 'predictions.csv'
        p = pd.read_csv(path, float_precision='round_trip')
        assert file_hash(path) == r['prediction_hash']
        assert array_hash(p.txId.to_numpy()) == r['query_ids_hash']
        recalculated = metrics(p.label, p.prediction)
        for k in ['auc', 'ap']:
            if recalculated[k] is not None:
                assert abs(recalculated[k] - r['metrics'][k]) < 1e-12
        s = r['spec']
        if s['kind'] == 'time':
            assert p.time_step.between(s['cut'] + 1, s['cut'] + s['horizon']).all()
        rows.append(dict(model=s['model'], variant=s['variant'], cut=s['cut'], seed=s['seed'], passed=True))
    return pd.DataFrame(rows)


In [ ]:
P = profiles(PROFILE)


## 3. Load matching completed experiments
Only use records produced with the same input files, source version and profile.


In [ ]:
data_hashes = {name: file_hash(DATA_DIR / name) for name in CSV_FILES}
data_identity = fingerprint(data_hashes)
records = [record for record in prediction_records(OUT)
    if record['spec']['data'] == data_identity and record['spec']['profile'] == P]
print('Completed matching trials:', len(records))
if not records:
    print('No matching results. Run notebooks 02-04 with the same paths, run name and profile.')


## 4. Prediction checks and window metrics


In [ ]:
if records:
    checks = audit_predictions(records)
    checks.to_csv(OUT / 'prediction_audit.csv', index=False)
    table = aggregate_runs(records)
    table.to_csv(OUT / 'window_metrics.csv', index=False)
    display(checks); display(table)
    # Exclude incomplete seed groups from the summary.
    complete = table.loc[table.complete].copy()
    keys = ['stage','model','variant','config_id','inner_horizon','kind','scaler','horizon','base']
    summary = complete.groupby(keys, dropna=False).agg(mean_auc=('auc','mean'),
        worst_auc=('auc','min'),mean_ap=('ap','mean'),windows=('cut','nunique')).reset_index()
    rolling_stages = ['rolling_models','training_policy','tabm_grids','tabm_regularization','tabm_features','graph_features']
    summary['expected_windows'] = summary.stage.map(lambda stage: len(P['cutoffs']) if stage in rolling_stages else np.nan)
    summary['complete_panel'] = summary.windows == summary.expected_windows
    summary.to_csv(OUT / 'summary.csv', index=False)
    display(summary.sort_values('mean_auc', ascending=False))


## 5. Per-period and first/next-seven-period metrics
Average matching seed predictions first. AUC is left empty when a period contains only one class. Keep each overlapping validation window separate.


In [ ]:
if records:
    groups = {}
    for record in records:
        key = fingerprint({k:v for k,v in record['spec'].items() if k != 'seed'})
        groups.setdefault(key, []).append(record)
    period_rows, half_rows = [], []
    for group in groups.values():
        spec = group[0]['spec']
        if sorted(row['spec']['seed'] for row in group) != sorted(P['seeds']):
            continue
        reference, parts = None, []
        for record in sorted(group, key=lambda row: row['spec']['seed']):
            frame = pd.read_csv(Path(record['path']) / 'predictions.csv', float_precision='round_trip')
            if reference is None:
                reference = frame[['txId','time_step','label']].copy()
            else:
                np.testing.assert_array_equal(reference, frame[['txId','time_step','label']])
            parts.append(frame.prediction.to_numpy())
        frame = reference.assign(prediction=np.mean(parts, axis=0))
        info = dict(stage=spec['stage'],model=spec['model'],variant=spec['variant'],cut=spec['cut'],
            config_id=fingerprint(spec['config'])[:10],inner_horizon=spec['inner_horizon'],horizon=spec['horizon'])
        for period, subset in frame.groupby('time_step'):
            period_rows.append(dict(**info,time_step=int(period),**metrics(subset.label,subset.prediction)))
        if spec['kind'] == 'time':
            for part, mask in [('first7',frame.time_step <= spec['cut']+7),('next7',frame.time_step > spec['cut']+7)]:
                subset = frame.loc[mask]
                if len(subset):
                    half_rows.append(dict(**info,part=part,**metrics(subset.label,subset.prediction)))
    periods = pd.DataFrame(period_rows); halves = pd.DataFrame(half_rows)
    periods.to_csv(OUT / 'per_period_metrics.csv', index=False)
    halves.to_csv(OUT / 'first_next7_metrics.csv', index=False)
    display(periods.head(20)); display(halves.head(20))


## 6. Reported results for reference
These are historical results from the final report, not scores computed by this run. Local validation AUC cannot determine a new Kaggle Public AUC.


In [ ]:
history = pd.DataFrame([
    ['Original TabM',.953378,.915344], ['Categorical feat_3',.952425,.909113],
    ['Joint upstream deviation and coverage',.953480,.914760],
    ['Dropout 0.15',.956330,.923081], ['Longer internal selection',.962427,.931294]],
    columns=['reported_recipe','reported_mean_auc','reported_worst_auc'])
display(history)
public_history = pd.DataFrame([
    dict(method='Optimized TabM',public_auc=.96075,exact_winning_artifact_available=False),
    dict(method='TabM + tree stacking',public_auc=.95173,exact_winning_artifact_available=True)])
display(public_history)
save_json(OUT / 'historical_public_scores.json', dict(source='Final report and user-confirmed Kaggle results',
    records=public_history.to_dict('records'),this_run_kaggle_auc=None))
